# Session 2b — Building agents with Python SDKs: ReAct, Plan-and-Execute, and Reflection Agent Paradigms

In Session 2a the Data Analyst could only *plan*, from a profile computed in advance. Here we rebuild it in Python with **LangChain and LangGraph**, give it tools that run real analyses on our 1,700 patients, and ask it to build a prediction model. Session 2a asked *which risk factors matter and which could we use to predict heart failure*. This module follows up:

> **Build the best model we can to predict in-hospital heart failure at admission. Choose the predictors and the type of model our data can support, and explain how well it predicts and which risk factors drive it.**

We give each pattern the **same model, the same four tools, and the same question**. Only the *control flow* differs, so any difference in behavior comes from the pattern.

| Pattern | The idea | Strength | Risk |
|---|---|---|---|
| **ReAct** | One loop: think → call a tool → observe → repeat | Adapts as it learns | Can wander; nothing forces it to look at the data first |
| **Plan-and-Execute** | Write a plan, run it step by step, replan when needed. **SRPA** adds a *Sense* step: look at the data before planning | Predictable, and the plan can be reviewed | A plan made blind can be generic; more model calls |
| **Reflection** | Draft → critique against a rubric → revise | Catches mistakes | Only as good as the rubric; more model calls |

**By the end you should be able to:**
- Build each pattern in LangChain or LangGraph and read its trace.
- Explain what the *Sense* step of SRPA adds to Plan-and-Execute.
- Compare the patterns on cost and on the quality of the answer, and choose one for a task.
---

## Setup - Before running the notebook

Install the packages once (Python 3.10 or newer):

```bash
pip install -r requirements.txt
```

**Set up your model with a `.env` file.** All settings live in one file next to the notebooks, so nothing secret is typed into a notebook:

1. Copy the example: `cp .env_example .env` (Windows: `copy .env_example .env`).
2. Open `.env`, set `PROVIDER` to the one you use, and fill in **only that provider's lines** (table below). Leave `MODEL` empty to use the default.
3. Restart the notebook kernel after editing `.env`.

Never share or commit `.env` (the `.gitignore` in this folder already excludes it). To get a key, follow the guides in `Instructions/`.

| `PROVIDER` | Fill in | Default `MODEL` |
|---|---|---|
| `anthropic` | `ANTHROPIC_API_KEY` | `claude-sonnet-5-5` |
| `openai` | `OPENAI_API_KEY` | `gpt-5.6-terra` |
| `foundry_claude` | `ANTHROPIC_FOUNDRY_API_KEY`, `ANTHROPIC_FOUNDRY_RESOURCE` | your deployment of `claude-sonnet-5-5` |
| `foundry_openai` | `AZURE_OPENAI_API_KEY`, `AZURE_OPENAI_BASE_URL` | your deployment of `gpt-5.6-terra` |

On Foundry, `MODEL` is your *deployment name*. `get_model()` in `mi_tools.py` reads the file (about five lines per provider). To try another model in one cell, pass it: `get_model("openai", "gpt-5.6-luna")`.

`REPLAY = True` shows the runs saved in `materials/runs/` instead of calling the model (no API calls), which is useful if the network is slow. Keep your `.env` filled in anyway: the model object is still created.

**Cost.** Each run also prints an *estimated* dollar cost (`cost_usd` in the comparison table), computed from the run's input and output tokens with [litellm](https://github.com/BerriAI/litellm)'s price list. These are list prices, so treat them as a rough comparison between patterns, not a bill. If your model is not in litellm's list, or litellm is not installed, the cost is left out. Runs saved before this was added have no cost until you run them again.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import operator
from typing import Annotated, Literal, TypedDict

import pandas as pd
from langchain.agents import create_agent
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field

from mi_tools import (ADMISSION, TOOLS, FinalModel, check_answer, compare_groups, get_model, list_variables,
                      profile_data, run_or_replay, show_run, show_trail, tool_sequence, trace_text, train_model)

REPLAY = False   # True = show the saved runs in materials/runs/ instead of calling the model

llm = get_model()    # PROVIDER, MODEL, and the keys come from your .env file (see above)
if not REPLAY:
    print(f"{type(llm).__name__} ({getattr(llm, 'model', None) or llm.model_name}):", llm.invoke("Reply with the single word: ready").text)

ChatAnthropic (claude-sonnet-5-5): ready


### The four tools

All three agents get the same fixed tools, ordinary Python functions from `mi_tools.py`. The agent sees only each tool's **name, description, and inputs**, so the descriptions say exactly what comes back. Run each tool once, with no model involved, to see what the agent sees:

| Tool | Input from the agent | Output back to the agent | The decision it informs |
|---|---|---|---|
| `list_variables` | `search` (text to look for, optional), `when_known` (`admission` or `any`), `max_missing_pct` (optional) | The number of matches, then one line per variable: name, type, when known, % missing, meaning | Which variables are allowed at admission |
| `profile_data` | `predictors`: a list of variable names | A short text: patients, events, predictors known only after admission, complete cases, **events per model column**, predictors with more than 30% missing | Is the data big and complete enough, and for what? |
| `compare_groups` | `variable`: one variable name; `cutoff` (optional split point, e.g. 70 for age) | That variable's unadjusted link with heart failure: rates per code with a chi-square test, or the mean with and without heart failure with a t-test, or rates below and above the cutoff with a risk ratio | Which risk factors to screen in |
| `train_model` | `predictors`: a list of variable names; `model_type` (`logistic`, `penalized_logistic`, `random_forest`, `gradient_boosting`); `missing_strategy` (`complete_case`, `impute`, `native`) | Cross-validated AUC, PR-AUC, Brier score, patients used, events per column, and the main drivers | Which model, and why |

In every case the agent sends only the inputs and gets back **plain text**. The cell below calls each tool by hand, so you can see the output exactly as the agent would.

In [2]:
print(list_variables.invoke({"search": "diabetes"}), "\n")
print(compare_groups.invoke({"variable": "age", "cutoff": 70}), "\n")
print(profile_data.invoke({"predictors": ["age", "sex", "diabetes", "serum_potassium"]}), "\n")
print(train_model.invoke({"predictors": ["age", "sex", "diabetes"], "model_type": "logistic", "missing_strategy": "complete_case"}))

1 variables (name | type | known at | missing | meaning):
diabetes | binary | admission | 0.6% missing | Diabetes mellitus in the anamnesis 

age: Age of patient. Patients with data: 1,692 (8 missing, left out).
  below 70: 1253 patients, 257 with heart failure (20.5%)
  70 or higher: 439 patients, 135 with heart failure (30.8%)
Chi-square test: p < 0.001.
Risk ratio (70 or higher vs below): 1.50. 

Patients: 1,700; heart failure events: 394 (23.2%).
Predictors: 4 (4 model columns after encoding categorical variables).
Known only after admission: none.
Complete cases (no missing predictor): 1,316 patients, 316 events.
Events per model column in complete cases: 79.0 (logistic regression usually needs about 10 or more).
Predictors with more than 30% missing: none.
Missing per predictor: age 0.5%, sex 0.0%, diabetes 0.6%, serum_potassium 21.8%. 

Model: logistic; missing data: complete_case; 3 predictors (3 model columns).
Patients used: 1,681 of 1,700 (392 events; 130.7 events per column

### The shared question and instructions

Every pattern uses the same question and the same short instructions. The instructions say *who the agent is* and *to use its tools for every number*. They do not say how to organize the work: that is what the pattern decides.

In [3]:
QUESTION = ("Build the best model we can to predict in-hospital heart failure at admission. Choose the predictors and the "
            "type of model our data can support, and explain how well it predicts and which risk factors drive it.")

ANALYST_PROMPT = """You are the Data Analyst on a research team studying heart failure after acute myocardial infarction (MI).
Our data: 1,700 patients admitted with acute MI to one hospital in Krasnoyarsk, Russia (1992-1995). The outcome is heart_failure, chronic heart failure during the hospital stay (394 patients, 23.2%).
Use your tools for every number. Never estimate or invent numbers. Report cross-validated results exactly as train_model reported them."""

---
## 1. ReAct: reason, act, observe

ReAct is one loop. The model **reasons** about what to do next, **acts** by calling a tool, **observes** the result, and repeats until it can answer. LangChain's `create_agent` is this loop; we give it the model, the tools, the instructions, and the shape of the final answer (`FinalModel`).

```
        ┌──────────────────────────────┐
        ▼                              │
   think ──► call a tool ──► see the result
        │
        └──► final answer
```

In [4]:
# create the REACT agent with the system prompt and the tools
react_agent = create_agent(llm, TOOLS, system_prompt=ANALYST_PROMPT, response_format=FinalModel)

# run the REACT agent with the question and return the messages, structured response, and summary
def run_react(stats):
    result = react_agent.invoke({"messages": [("user", QUESTION)]}, config={"callbacks": [stats], "recursion_limit": 50})
    return result["messages"], result["structured_response"], None

# run the REACT agent and show the messages, structured response, and summary
react_messages, react_final, react_summary, _ = run_or_replay("react", REPLAY, run_react)
show_run(react_messages, react_final, react_summary)

🔧 list_variables(when_known=admission)
   ↳ 102 variables (name | type | known at | missing | meaning): | age | integer | admission | 0.5% missing | Age of patient (years) | sex | binary | admission | 0.0% missing | Sex. Codes: 0=female; 1=male | prior_mi_count | ordinal | admission | 0.2% missing | Quantity of myocardial infarctions in the a
🔧 profile_data(predictors=['age', 'sex', 'prior_mi_count', 'angina_functional_class', 'hypertension_stage', 'prior_heart_failure', 'diabetes', 'obesity', 'pulmonary_edema_at_admission', 'cardiogenic_shock_at_admission', 'atrial_fibrillation_at_admission', 'anterior_mi', 'inferior_mi', 'lateral_mi', 'posterior_mi',)
🔧 train_model(predictors=['age', 'sex', 'prior_mi_count', 'angina_functional_class', 'hypertension_stage', 'prior_heart_failure', 'diabetes', 'obesity', 'pulmonary_edema_at_admission', 'cardiogenic_shock_at_admission', 'atrial_fibrillation_at_admission', 'anterior_mi', 'inferior_mi', 'lateral_mi', 'posterior_mi',)
🔧 train_model(predicto

Bad pipe message: %s [b'am\r\ntraceparent: 00-012956cb3f42a4e6c2f8b089bfc6fd97-966eeb7ebec20527-00\r\nContent-Type: application/json\r\nContent', b'ength: 212\r\n\r\n{"jsonrpc":"2.0","id":1,"metho']
Bad pipe message: %s [b':"initialize","params":{"protocolVersion":"2025-06-18","capabilities":{"tools":{},"resources":{},"p']


**What to look at.** 💭 is what the model said, 🔧 the tool it called, ↳ what came back.
- Where did it *start*: did it look at the data (`list_variables`, `profile_data`) before fitting a model?
- Which variables did it choose, and were they all known at admission?
- Which model type and missing-data strategy did it choose, and what did it base that on?
- Did it keep trying more models, or stop? What made it stop?

**The full trail.** The lines above are a summary. The cell below prints *every* model call and tool call of the run, in order: the prompts, the model's replies and tool requests, and each tool's result (cut at 500 characters here). The complete text, nothing cut, is saved next to the run in `materials/runs/react_trace.md`, so you can open it and compare the patterns side by side.

In [6]:
show_trail("react")


[1] MODEL CALL (model)
  💬 system: You are the Data Analyst on a research team studying heart failure after acute myocardial infarction (MI).
Our data: 1,700 patients admitted with acute MI to one hospital in Krasnoyarsk, Russia (1992-1995). The outcome is heart_failure, chronic heart failure during the hospital stay (394 patients, 23.2%).
Use your tools for every number. Never estimate or invent numbers. Report cross-validated results exactly as train_model reported them.
  💬 human: Build the best model we can to predict in-hospital heart failure at admission. Choose the predictors and the type of model our data can support, and explain how well it predicts and which risk factors drive it.
  🔧 calls list_variables({'when_known': 'admission'})

[2] TOOL RESULT: list_variables
  ↳ 102 variables (name | type | known at | missing | meaning):
age | integer | admission | 0.5% missing | Age of patient (years)
sex | binary | admission | 0.0% missing | Sex. Codes: 0=female; 1=male
prior_mi_co

---
### Optional: the same agent without `create_agent`

`create_agent` is not magic. It builds a small LangGraph graph with one **model node**, one **tool node**, and one decision: *did the model ask for a tool?* Below we build that graph ourselves, with the same model, tools, prompt, and answer shape.

```
 START ──► think ──► did the model ask for a tool? ──no──► answer ──► END
            ▲                       │
            │                      yes
            └────────── act ◄───────┘
```

- **think** calls the model with the conversation so far. **act** runs the tools it asked for and adds the results to the conversation. The loop is the edge from `act` back to `think`.
- **answer** is a last model call that puts what the model found into the `FinalModel` shape.

In [5]:
# the same ReAct agent, built by hand as a LangGraph graph: this is what create_agent does for us
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph.message import add_messages

class ReActState(TypedDict, total=False):
    messages: Annotated[list, add_messages]       # the conversation so far; each node appends to it
    final: FinalModel

llm_with_tools = llm.bind_tools(TOOLS)            # the model may now ask for any of our tools
answer_format = llm.with_structured_output(FinalModel)
TOOLS_BY_NAME = {tool.name: tool for tool in TOOLS}

def think(state, config):
    """Model node: reason about what to do next. The reply may contain tool calls."""
    return {"messages": [llm_with_tools.invoke([SystemMessage(ANALYST_PROMPT)] + state["messages"], config)]}

def act(state, config):
    """Tool node: run every tool the model asked for and add each result as a message."""
    calls = state["messages"][-1].tool_calls
    return {"messages": [TOOLS_BY_NAME[call["name"]].invoke(call, config) for call in calls]}

def keep_going(state):
    """The decision: tool calls -> act, no tool calls -> the model is done."""
    return "act" if state["messages"][-1].tool_calls else "answer"

def answer(state, config):
    """Final node: turn everything the model found into the FinalModel shape."""
    messages = [SystemMessage(ANALYST_PROMPT)] + state["messages"] + [HumanMessage("Give your final answer in the required format.")]
    return {"final": answer_format.invoke(messages, config)}

builder = StateGraph(ReActState)
builder.add_node("think", think)
builder.add_node("act", act)
builder.add_node("answer", answer)
builder.add_edge(START, "think")
builder.add_conditional_edges("think", keep_going, {"act": "act", "answer": "answer"})
builder.add_edge("act", "think")                  # the loop: after acting, think again
builder.add_edge("answer", END)
scratch_agent = builder.compile()

# run it exactly like the create_agent version
def run_scratch(stats):
    result = scratch_agent.invoke({"messages": [("user", QUESTION)]}, config={"callbacks": [stats], "recursion_limit": 50})
    return result["messages"], result["final"], None

scratch_messages, scratch_final, scratch_summary, _ = run_or_replay("react_scratch", REPLAY, run_scratch)
show_run(scratch_messages, scratch_final, scratch_summary)

🔧 list_variables(when_known=admission)
   ↳ 102 variables (name | type | known at | missing | meaning): | age | integer | admission | 0.5% missing | Age of patient (years) | sex | binary | admission | 0.0% missing | Sex. Codes: 0=female; 1=male | prior_mi_count | ordinal | admission | 0.2% missing | Quantity of myocardial infarctions in the a
🔧 profile_data(predictors=['age', 'sex', 'prior_mi_count', 'angina_functional_class', 'hypertension_stage', 'prior_heart_failure', 'diabetes', 'obesity', 'systolic_bp_icu', 'diastolic_bp_icu', 'pulmonary_edema_at_admission', 'cardiogenic_shock_at_admission', 'atrial_fibrillation_at_admission', 'anterior_mi', 'infe)
🔧 compare_groups(variable=pulmonary_edema_at_admission)
🔧 compare_groups(variable=prior_heart_failure)
   ↳ Patients: 1,700; heart failure events: 394 (23.2%). | Predictors: 21 (24 model columns after encoding categorical variables). | Known only after admission: none. | Complete cases (no missing predictor): 784 patients, 189 events. |

**What to notice.** The trail has the same shape as the `create_agent` run above: think, act, think, act, until the model stops asking for tools. The exact tools and numbers can differ between runs, because the model chooses them.

`create_agent` does more than our 25 lines: it can run several tool calls at the same time (ours run one after another), it reports a tool error back to the model instead of stopping, and it has hooks (called *middleware*) for things like asking a human before a tool runs. The core is still the loop above.

---
## 2. Plan-and-Execute, and SRPA

Plan-and-Execute separates *deciding what to do* from *doing it*. This is a LangGraph graph with a few nodes:

```
 plan ──► execute one step ──► replan ──► more to do? ──yes──► execute one step
 (LLM)    (a ReAct agent)      (LLM)           │
                                               no
                                               ▼
                                         final answer
```

The **plan** is a list you can read (and, in a real project, approve) before anything runs. After each step, **replan** decides whether to continue, change the remaining steps, or finish.

**SRPA (Sense – Reason – Plan – Act)** adds two steps *before* planning:

| Step | What happens | Uses an LLM? |
|---|---|---|
| **Sense** | Look at the data: one call to `profile_data` on every variable known at admission | No, just a tool call |
| **Reason** | Write a short assessment of what that means for modelling | Yes |
| **Plan** | Write the plan, now with the assessment in front of it | Yes |
| **Act** | Execute the steps | Yes, with tools |

We build one graph and switch the Sense and Reason steps on or off, so the only difference between the two runs is whether the planner has looked at the data.

In [6]:
# build a plan of steps to answer the question, then execute them one by one, replanning if needed
class Plan(BaseModel):
    steps: list[str] = Field(description="3 to 5 concrete steps, each doable with the tools")

# replan if the results so far are not enough to answer the question
class Replan(BaseModel):
    done: bool = Field(description="True if the results so far are enough to answer the question")
    remaining_steps: list[str] = Field(description="Steps still to do; empty if done")

# the final answer after all steps are done
class PlanState(TypedDict, total=False):
    sensed: str                                          # Sense: facts about the data
    assessment: str                                      # Reason: what they mean for modelling
    first_plan: list[str]
    todo: list[str]                                      # steps still to do
    past_steps: Annotated[list[tuple[str, str]], operator.add]
    trace: Annotated[list, operator.add]                 # every tool call, for show_run
    final: FinalModel

step_agent = create_agent(llm, TOOLS, system_prompt=ANALYST_PROMPT)      # a ReAct agent that does ONE step
make_plan = llm.with_structured_output(Plan)
make_replan = llm.with_structured_output(Replan)
make_final = llm.with_structured_output(FinalModel)

TOOL_LIST = "\n".join(f"- {tool.name}: {tool.description.splitlines()[0]}" for tool in TOOLS)      # what the planner may use

# return a string summary of the steps done so far, for the planner to reason about
def history(state):
    return "\n".join(f"- {step}: {result}" for step, result in state.get("past_steps", [])) or "nothing yet"

# return a string summary of the tool calls and results, for the final answer
def sense(state):
    """Sense: look at the data before planning. No LLM, just a tool call."""
    return {"sensed": profile_data.invoke({"predictors": ADMISSION})}

# return a string summary of what the sensed data means for modelling, for the planner to reason about
def reason(state, config):
    """Reason: what does what we sensed mean for the modelling?"""
    reply = llm.invoke(f"{ANALYST_PROMPT}\n\nQuestion: {QUESTION}\n\nWhat we sensed about the data (every variable known at admission):\n{state['sensed']}\n\n"
                       "In 4-6 short lines, assess what this data can support: how many predictors, how to handle missing data, which model types are realistic.", config)
    return {"assessment": reply.text}

# return a plan of 3 to 5 steps to answer the question, each doable with the tools
def plan(state, config):
    context = f"\nWhat we know about the data:\n{state['sensed']}\n{state['assessment']}\n" if "assessment" in state else ""
    steps = make_plan.invoke(f"{ANALYST_PROMPT}\n\nQuestion: {QUESTION}\n{context}\nTools you can use:\n{TOOL_LIST}\n\n"
                             "Write a plan of 3 to 5 steps to answer the question. Each step must be doable with these tools.", config).steps
    return {"todo": steps, "first_plan": steps}

# execute the next step in the plan, using the tools, and return the result
def execute(state, config):
    step = state["todo"][0]
    result = step_agent.invoke({"messages": [("user", f"Question: {QUESTION}\n\nResults so far:\n{history(state)}\n\n"
                                                      f"Carry out ONLY this step: {step}\nReply with the key facts, with numbers exactly as the tools returned them.")]},
                               config={**config, "recursion_limit": 25})
    return {"past_steps": [(step, result["messages"][-1].text)], "todo": state["todo"][1:], "trace": result["messages"]}

# replan if the results so far are not enough to answer the question, returning the steps still needed
def replan(state, config):
    reply = make_replan.invoke(f"{ANALYST_PROMPT}\n\nQuestion: {QUESTION}\n\nResults so far:\n{history(state)}\n\nSteps still planned:\n" + "\n".join(state["todo"]) +
                               "\n\nAre the results enough to answer the question? If not, list the steps still needed (you may change the planned steps; each must be doable with the tools).", config)
    return {"todo": [] if reply.done else reply.remaining_steps}

# return the final answer after all steps are done, including the best model trained and its performance
def finish(state, config):
    return {"final": make_final.invoke(f"{ANALYST_PROMPT}\n\nQuestion: {QUESTION}\n\nResults of all steps:\n{history(state)}\n\n"
                                       f"Every tool call and result:\n{trace_text(state['trace'], 800)}\n\n"
                                       "Give the final answer for the best model that was actually trained. Use exact variable names and only numbers that appear in these results.", config)}

# return "execute" if there are still steps to do and we haven't done 6 steps yet, else "finish"
def next_step(state):
    return "execute" if state["todo"] and len(state["past_steps"]) < 6 else "finish"        # at most 6 steps

# build a state graph of the planner, with or without the sense and reason steps
def build_planner(with_sense):
    graph = StateGraph(PlanState)
    for node in (plan, execute, replan, finish):
        graph.add_node(node.__name__, node)
    if with_sense:                                       # SRPA: Sense -> Reason -> Plan -> Act
        graph.add_node("sense", sense)
        graph.add_node("reason", reason)
        graph.add_edge(START, "sense")
        graph.add_edge("sense", "reason")
        graph.add_edge("reason", "plan")
    else:
        graph.add_edge(START, "plan")
    graph.add_edge("plan", "execute")
    graph.add_edge("execute", "replan")
    graph.add_conditional_edges("replan", next_step, ["execute", "finish"])
    graph.add_edge("finish", END)
    return graph.compile()

# run the planner with or without the sense and reason steps, returning the trace, final answer, and any other state
def run_planner(with_sense):
    def run(stats):
        state = build_planner(with_sense).invoke({}, config={"callbacks": [stats], "recursion_limit": 60})
        return state["trace"], state["final"], {key: state.get(key) for key in ("sensed", "assessment", "first_plan", "past_steps")}
    return run

### Plan-and-Execute without a Sense step

The planner writes its plan from the question alone.

In [7]:
plan_messages, plan_final, plan_summary, plan_extra = run_or_replay("plan", REPLAY, run_planner(with_sense=False))
print("THE PLAN (written without looking at the data):")
for i, step in enumerate(plan_extra["first_plan"], 1):
    print(f"  {i}. {step}")
print()
show_run(plan_messages, plan_final, plan_summary)

THE PLAN (written without looking at the data):
  1. Call list_variables to see every variable, when it is known (admission vs. later in the stay), % missing and meaning. Exclude any variable known only after admission, such as in-hospital complications, treatments given during the stay, and outcomes. These would leak the outcome.
  2. Call profile_data on the candidate admission-time predictors. Check patients with complete data, events (394) and events per variable. This decides how many predictors and what model complexity the data can support, and flags variables with heavy missingness to drop or handle.
  3. Call compare_groups on the key candidate predictors one at a time (age, sex, prior MI or angina, hypertension, blood pressure, potassium, ECG location, etc.; use cutoffs for the continuous ones). This is for screening and sanity checks only. The unadjusted results are not used to pick the final model.
  4. Call train_model with 5-fold stratified CV on the chosen admission-time

In [8]:
# Every model call and tool call, in order: plan, then each step and replan.
show_trail("plan")


[1] MODEL CALL (plan)
  💬 human: You are the Data Analyst on a research team studying heart failure after acute myocardial infarction (MI).
Our data: 1,700 patients admitted with acute MI to one hospital in Krasnoyarsk, Russia (1992-1995). The outcome is heart_failure, chronic heart failure during the hospital stay (394 patients, 23.2%).
Use your tools for every number. Never estimate or invent numbers. Report cross-validated results exactly as train_model reported them.

Question: Build the best model we can to predict in-hosp ...
  🔧 calls Plan({'steps': ['Call list_variables to see every variable, when it is known (admission vs. later in the stay), % missing and meaning. Exclude any variable known only after admission, such as in-hospital complications, treatments given during the stay, and outcomes. These would leak the outcome.', 'Call profile_data on the candidate admission-time predictors. Check patients with complete data, events (394) and events per variable. This decides how

### SRPA: Sense – Reason – Plan – Act

Now the graph first **senses** the data and **reasons** about it, and only then plans.

In [9]:
srpa_messages, srpa_final, srpa_summary, srpa_extra = run_or_replay("srpa", REPLAY, run_planner(with_sense=True))
print("SENSE (no model, one tool call):\n" + srpa_extra["sensed"])
print("\nREASON (one model call):\n" + srpa_extra["assessment"])
print("\nTHE PLAN (written after sensing and reasoning):")
for i, step in enumerate(srpa_extra["first_plan"], 1):
    print(f"  {i}. {step}")
print()
show_run(srpa_messages, srpa_final, srpa_summary)

SENSE (no model, one tool call):
Patients: 1,700; heart failure events: 394 (23.2%).
Predictors: 102 (106 model columns after encoding categorical variables).
Known only after admission: none.
Complete cases (no missing predictor): 0 patients, 0 events.
Events per model column in complete cases: 0.0 (logistic regression usually needs about 10 or more).
Predictors with more than 30% missing: family_history_coronary_disease (95.8%), systolic_bp_emergency_team (63.3%), diastolic_bp_emergency_team (63.3%), serum_cpk (99.8%), opioids_emergency_team (38.6%), nsaids_emergency_team (40.4%), lidocaine_emergency_team (39.8%).

REASON (one model call):
Nothing has been modeled yet, so these are design judgments from the counts you gave. The only arithmetic is 394 events ÷ 106 columns.

- **Predictor count:** With 394 events and 106 model columns, we have about 3.7 events per column, well below the ~10 rule of thumb. Unpenalized logistic regression on everything is not supported. At 10 events per 

In [10]:
# Every model call and tool call, in order: sense, reason, plan, then each step and replan.
show_trail("srpa")


[1] TOOL RESULT: profile_data
  ↳ Patients: 1,700; heart failure events: 394 (23.2%).
Predictors: 102 (106 model columns after encoding categorical variables).
Known only after admission: none.
Complete cases (no missing predictor): 0 patients, 0 events.
Events per model column in complete cases: 0.0 (logistic regression usually needs about 10 or more).
Predictors with more than 30% missing: family_history_coronary_disease (95.8%), systolic_bp_emergency_team (63.3%), diastolic_bp_emergency_team (63.3%), serum_cpk (99.8%), opioid ...

[2] MODEL CALL (reason)
  💬 human: You are the Data Analyst on a research team studying heart failure after acute myocardial infarction (MI).
Our data: 1,700 patients admitted with acute MI to one hospital in Krasnoyarsk, Russia (1992-1995). The outcome is heart_failure, chronic heart failure during the hospital stay (394 patients, 23.2%).
Use your tools for every number. Never estimate or invent numbers. Report cross-validated results exactly as train_mo

**What to look at.** Compare the two plans above.
- Is the plan *without* sensing generic ("impute, try a few models")? Does the SRPA plan name specific variables, a missing-data strategy, and a model type, because it knows what the data can support?
- Did the replanner change the plan along the way?
- What did the extra Sense and Reason steps cost, compared with their effect on the answer?

---
## 3. Reflection: draft, critique, revise

Reflection adds a second role, a **critic**, that checks the work against a rubric. A LangGraph loop runs: the analyst drafts an answer (the ReAct agent from section 1), the critic reviews it, and if the critic is not satisfied the analyst revises, at most twice.

```
 draft ──► critique ──accept──► done
              │
            revise ──► (back to critique)
```

The critic sees the answer **and the analyst's tool calls and results**, so it can check that every number is real. The rubric extends the review checklist from Module 1 with modelling rules.

In [11]:
class Critique(BaseModel):
    verdict: Literal["accept", "revise"]
    issues: list[str] = Field(description="Specific problems to fix; empty if accepted")

RUBRIC = """Check the analyst's answer against these rules:
1. Only variables known at admission are used as predictors. A prediction made at admission cannot use data from hospital days 1-3.
2. Missing data is handled sensibly: no predictor with more than 30% missing values unless the model handles missing values natively, and the strategy fits the model.
3. There are enough events per model column for the chosen model (about 10 or more for logistic regression).
4. AUC is reported together with PR-AUC (heart failure is 23% of patients), and every number appears in a tool result in the evidence.
5. The model type is justified by what the data can support, not only by the best score.
6. Cardiogenic shock appears to protect against heart failure because those patients die early. If it is used, this competing risk must be mentioned."""

class ReflectionState(TypedDict, total=False):
    final: FinalModel
    trace: Annotated[list, operator.add]
    drafts: Annotated[list, operator.add]                # every version of the answer
    review: Critique
    reviews: Annotated[list, operator.add]               # every critique
    rounds: int

critic = llm.with_structured_output(Critique)

def draft(state, config):
    result = react_agent.invoke({"messages": [("user", QUESTION)]}, config={**config, "recursion_limit": 40})
    return {"final": result["structured_response"], "trace": result["messages"], "drafts": [result["structured_response"].model_dump()], "rounds": 0}

def critique(state, config):
    review = critic.invoke(f"You are a careful reviewer of data analyses.\n\n{RUBRIC}\n\nQuestion: {QUESTION}\n\nAnalyst's answer:\n{state['final'].model_dump_json(indent=2)}\n\n"
                           f"Evidence (the analyst's tool calls and results):\n{trace_text(state['trace'])}\n\n"
                           "List only rules that are actually broken, at most 5 issues, one sentence each with the specifics. If no rule is broken, accept with no issues.", config)
    return {"review": review, "reviews": [review.model_dump()]}

def revise(state, config):
    issues = "\n".join(f"- {issue}" for issue in state["review"].issues)
    result = react_agent.invoke({"messages": [("user", f"{QUESTION}\n\nYour previous answer:\n{state['final'].model_dump_json(indent=2)}\n\n"
                                                      f"A reviewer found these problems:\n{issues}\n\nFix them (refit the model if needed) and give an improved final answer.")]},
                                config={**config, "recursion_limit": 40})
    return {"final": result["structured_response"], "trace": result["messages"], "drafts": [result["structured_response"].model_dump()], "rounds": state["rounds"] + 1}

def after_critique(state):
    return END if state["review"].verdict == "accept" or state["rounds"] >= 2 else "revise"     # at most 2 revisions

graph = StateGraph(ReflectionState)
graph.add_node("draft", draft)
graph.add_node("critique", critique)
graph.add_node("revise", revise)
graph.add_edge(START, "draft")
graph.add_edge("draft", "critique")
graph.add_conditional_edges("critique", after_critique, ["revise", END])
graph.add_edge("revise", "critique")
reflection_graph = graph.compile()

In [12]:
def run_reflection(stats):
    state = reflection_graph.invoke({}, config={"callbacks": [stats], "recursion_limit": 60})
    return state["trace"], state["final"], {"drafts": state["drafts"], "reviews": state["reviews"]}

reflection_messages, reflection_final, reflection_summary, reflection_extra = run_or_replay("reflection", REPLAY, run_reflection)

for i, (answer, review) in enumerate(zip(reflection_extra["drafts"], reflection_extra["reviews"])):
    print(f"VERSION {i + 1}: {answer['model_type']} ({answer['missing_strategy']}), {len(answer['predictors'])} predictors, reported AUC {answer['cv_auc']:.3f}")
    print(f"  CRITIC: {review['verdict'].upper()}" + "".join(f"\n   - {issue}" for issue in review["issues"]))
print()
show_run(reflection_messages, reflection_final, reflection_summary)

VERSION 1: penalized_logistic (impute), 22 predictors, reported AUC 0.729
  CRITIC: ACCEPT

🔧 list_variables(when_known=admission)
   ↳ 102 variables (name | type | known at | missing | meaning): | age | integer | admission | 0.5% missing | Age of patient (years) | sex | binary | admission | 0.0% missing | Sex. Codes: 0=female; 1=male | prior_mi_count | ordinal | admission | 0.2% missing | Quantity of myocardial infarctions in the a
🔧 profile_data(predictors=['age', 'sex', 'prior_mi_count', 'angina_functional_class', 'hypertension_stage', 'prior_heart_failure', 'diabetes', 'obesity', 'atrial_fibrillation_at_admission', 'cardiogenic_shock_at_admission', 'pulmonary_edema_at_admission', 'anterior_mi', 'inferior_mi', 'lateral_mi', 'posterior_mi',)
🔧 compare_groups(variable=pulmonary_edema_at_admission)
🔧 compare_groups(variable=prior_heart_failure)
   ↳ Patients: 1,700; heart failure events: 394 (23.2%). | Predictors: 22 (25 model columns after encoding categorical variables). | Known only

In [13]:
# Every model call and tool call, in order: the draft, the critic's review, and any revision.
show_trail("reflection")


[1] MODEL CALL (draft > model)
  💬 system: You are the Data Analyst on a research team studying heart failure after acute myocardial infarction (MI).
Our data: 1,700 patients admitted with acute MI to one hospital in Krasnoyarsk, Russia (1992-1995). The outcome is heart_failure, chronic heart failure during the hospital stay (394 patients, 23.2%).
Use your tools for every number. Never estimate or invent numbers. Report cross-validated results exactly as train_model reported them.
  💬 human: Build the best model we can to predict in-hospital heart failure at admission. Choose the predictors and the type of model our data can support, and explain how well it predicts and which risk factors drive it.
  🔧 calls list_variables({'when_known': 'admission'})

[2] TOOL RESULT: list_variables
  ↳ 102 variables (name | type | known at | missing | meaning):
age | integer | admission | 0.5% missing | Age of patient (years)
sex | binary | admission | 0.0% missing | Sex. Codes: 0=female; 1=male
pri

**What to look at.**
- What did the critic object to? Were those real problems or nitpicks?
- Did the revised answer actually fix them, or only reword them?
- The critic is only as good as its rubric: what would it *miss*? (It cannot flag a problem that is not on the list.)

---
## 4. Comparing the patterns

The table puts the four runs side by side: what they cost, and how good the answer is. The quality columns do **not** trust what the agent claimed. For each final answer, `check_answer` re-runs the chosen model with the same tool and checks simple rules.

| Column | Meaning |
|---|---|
| `reported AUC` / `verified AUC` | What the agent said, and what the tool gives when we re-run its chosen model |
| `reported_auc_matches` | The two agree (within 0.02) |
| `admission_only` | Every predictor is known at admission |
| `no_variable_over_30pct_missing` | No predictor with more than 30% missing |
| `events_per_column_at_least_10` | Enough events for the number of model columns |

In [14]:
runs = {"ReAct": (react_final, react_summary), "Plan-and-Execute": (plan_final, plan_summary),
        "SRPA": (srpa_final, srpa_summary), "Reflection": (reflection_final, reflection_summary)}
rows = []
for name, (final, summary) in runs.items():
    rows.append({"pattern": name, **summary, "model": final.model_type, "predictors": len(final.predictors),
                 "reported AUC": final.cv_auc, **check_answer(final)})
table = pd.DataFrame(rows).set_index("pattern").drop(columns=["valid", "input_tokens", "output_tokens"], errors="ignore")
table.replace({True: "✅", False: "❌"})

,llm_calls,tool_calls,tokens,cost_usd,seconds,model,predictors,reported AUC,verified_auc,reported_auc_matches,admission_only,no_variable_over_30pct_missing,events_per_column_at_least_10
pattern,,,,,,,,,,,,,
ReAct,4,6,36158,0.096292,24,logistic,12,0.732,0.732,✅,✅,✅,✅
Plan-and-Execute,25,14,228297,0.604530,125,penalized_logistic,26,0.737,0.737,✅,✅,✅,✅
SRPA,26,19,238233,0.681754,189,penalized_logistic,76,0.724,0.724,✅,✅,✅,❌
Reflection,5,7,39989,0.107570,33,penalized_logistic,22,0.729,0.729,✅,✅,✅,✅


### The order of tool calls

The saved trails also show how differently the patterns *spend their effort*. This is the order of tool calls in each run (`×3` means three calls in a row):

In [ ]:
for name, label in (("react", "ReAct"), ("plan", "Plan-and-Execute"), ("srpa", "SRPA"), ("reflection", "Reflection")):
    print(f"{label}:\n  {tool_sequence(name)}\n")

ReAct:
  list_variables → profile_data → train_model ×4

Plan-and-Execute:
  list_variables ×3 → profile_data ×3 → list_variables → train_model → list_variables → train_model → list_variables → train_model → list_variables → train_model

SRPA:
  profile_data → list_variables ×2 → profile_data → list_variables → train_model → profile_data → train_model ×5 → list_variables ×3 → train_model → list_variables → train_model → compare_groups

Reflection:
  list_variables → compare_groups ×2 → profile_data → train_model ×3



Bad pipe message: %s [b'am\r\ntraceparent: 00-5921fe533b37c4d5db8fe8426dcc43b7-b8a2a2130a20cffc-00\r\nContent-Type: application/json\r\nContent', b'ength: 212\r\n\r\n{"jsonrpc":"2.0","id":1,"metho']
Bad pipe message: %s [b':"initialize","params":{"protocolVersion":"2025-06-18","capabilities":{"tools":{},"resources":{},"p']
Bad pipe message: %s [b'am\r\ntraceparent: 00-915d4697cf254886d3df7ec670f1a3c8-e07c506e6fff3914-00\r\nContent-Type: application/json\r\nContent', b'ength: 212\r\n\r\n{"jsonrpc":"2.0","id":1,"metho']
Bad pipe message: %s [b':"initialize","params":{"protocolVersion":"2025-06-18","capabilities":{"tools":{},"resources":{},"p']
Bad pipe message: %s [b'am\r\ntraceparent: 00-0247d90e1235ba6881fd134e4110f54b-8982a40dbc31e898-00\r\nContent-Type: application/json\r\nContent', b'ength: 212\r\n\r\n{"jsonrpc":"2.0","id":1,"metho']
Bad pipe message: %s [b':"initialize","params":{"protocolVersion":"2025-06-18","capabilities":{"tools":{},"resources":{},"p']
Bad pipe message: %s [b'a

**Read the table with care.** One run of each pattern is an anecdote: language models give different answers on different runs, and a different model gives a different table. Re-run a pattern and see what changes. What does stay true is *how* each pattern spends its effort: ReAct is cheapest, Plan-and-Execute and SRPA make many more model calls, and Reflection costs extra only when the critic sends the analyst back to revise.

Questions to ask of your own table:
- Which answer would you want to defend in a paper, and which check settles that, not the AUC alone?
- Did SRPA's plan *use* what it sensed (events per column, missing data)? Did the executor stick to the plan?
- Did the critic find a real problem, or accept too easily? What would a stricter rubric add?

### Which pattern when?

| | Use it when | Watch out for |
|---|---|---|
| **ReAct** | The task is short or exploratory, and you cannot say in advance what the steps are | It may skip the careful steps (looking at the data first) unless you ask for them |
| **Plan-and-Execute** | The task has many steps, you want a plan to review or follow, or you want to use a cheaper model for the steps | A plan made without data can be generic; add **Sense** so it starts from facts |
| **Reflection** | Quality matters and you can write down what "good" means as a rubric | The critic only catches what the rubric lists; each round costs more |

They also combine: *plan* the work, *execute* each step with a ReAct agent (as in section 2), then *reflect* on the result. That composite is a good take-home exercise.

### Discussion
1. Which pattern would you trust for an analysis that will go into a paper, and why?
2. The critic used a rubric that *we* wrote. Where would the rubric come from in your own project?
3. What would you add to the Sense step for a dataset with repeated measurements per patient?

### Next
Module 3 puts these roles in a team: a planner, an analyst, a critic, and a reviewer as **separate agents** that hand work to each other.

---
## Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `ValueError: ... is not set` | A key or setting for your `PROVIDER` is missing | Copy `.env_example` to `.env`, fill in the lines for your provider, and restart the kernel |
| Changes to `.env` have no effect | `.env` is read when the model is created, and values already set in your shell win | Restart the kernel; check that your shell does not set the same variable |
| `tool_choice: type "tool" and "any" are not supported for this model` (a 400 error) | Your `langchain-anthropic` is older than the notebook was tested with, so it does not know the model and asks for a forced tool call that newer Claude models reject | Run `pip install -U -r requirements.txt` in the environment of your kernel and restart it; or, for a quick test, set `MODEL=claude-haiku-5-5` in `.env` |
| 401 / authentication error | Wrong or expired key | Re-copy the key and restart the notebook kernel |
| 404 / model or deployment not found | `MODEL` is not a model your key can use (on Foundry: not a deployment name) | Use an exact model or deployment name |
| 429 / rate limit | Too many requests, for example a whole room on one key | Wait a few seconds and re-run; or use `REPLAY = True` |
| `GraphRecursionError` | The agent kept looping | Raise `recursion_limit`, or check the tool errors in the trace |
| A run takes minutes | A model call or a random forest is slow | Normal for ReAct and Reflection; use `REPLAY = True` to see saved runs |